# 06 · Benchmark de escalabilidad

Objetivo: medir el motor local y comprobar la recuperación de eventos ficticios. La escala inicial es de 1.000 filas. Es posible seleccionar 10.000 o 100.000 según los recursos.

**Requisitos.** Python, ZIP y dependencias fijadas. Los eventos carecen de personas y coordenadas. Todas las observaciones del banco están etiquetadas [SIMULADO].

In [ ]:
from pathlib import Path
import importlib.metadata as metadata
import os
import subprocess
import sys
import tempfile
import zipfile

package_name = "SLP-BigData-Reproducible"
here = Path.cwd().resolve()
explicit = os.getenv("SLP_BIGDATA_ROOT")
candidates = [Path(explicit).expanduser().resolve()] if explicit else [
    here, here.parent, here / package_name,
]
ROOT = next((p for p in candidates if (p / "src" / "repro.py").is_file()), None)
colab = "COLAB_RELEASE_TAG" in os.environ or "google.colab" in sys.modules
if ROOT is None and colab:
    from google.colab import files
    print("Selecciona el ZIP SLP-BigData-Reproducible.zip.")
    uploaded = files.upload()
    if set(uploaded) != {package_name + ".zip"}:
        raise ValueError("Selecciona únicamente el ZIP de distribución del proyecto.")
    with zipfile.ZipFile(package_name + ".zip") as archive:
        for entry in archive.infolist():
            parts = Path(entry.filename).parts
            if not parts or parts[0] != package_name or ".." in parts or "\\" in entry.filename:
                raise ValueError("El ZIP contiene una ruta ajena al paquete.")
        archive.extractall(here)
    ROOT = here / package_name
if ROOT is None or not (ROOT / "src" / "repro.py").is_file():
    raise FileNotFoundError("Abre la carpeta descomprimida o define SLP_BIGDATA_ROOT.")

requirements = ROOT / "requirements.txt"
def current_version(name):
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return None
pins = [line.split("==", 1) for line in requirements.read_text(encoding="utf-8").splitlines()
        if "==" in line and not line.startswith("#")]
missing = [(name, version) for name, version in pins if current_version(name) != version]
if missing:
    if not colab:
        raise RuntimeError("Instala requirements.txt en el entorno virtual activo antes de continuar.")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)], check=True)
sys.path.insert(0, str(ROOT))
from src.repro import run_demo, run_original, run_benchmark, query, read_report

private = ROOT / ".bigdata-private"
private.mkdir(exist_ok=True)
WORK = Path(tempfile.mkdtemp(prefix="notebook-", dir=private))
print("Dependencias verificadas:", {name: current_version(name) for name, _ in pins})

In [ ]:
FILAS = 1000
assert FILAS in {1000, 10000, 100000}
resultado = run_benchmark(WORK / "benchmark", rows=FILAS)
print("[SIMULADO]", resultado)

**Resultado esperado.** El número de eventos únicos se conserva tras deduplicación y reingesta. El reporte expone tiempo, tamaños, consultas y hashes. Los tiempos dependen de la máquina, la cardinalidad y el estado del sistema. [Benchmark histórico](../docs/resultados_verificados.md).